# StoneAge: Pet World — Performance Investigation

**Client:** Netmarble  
**Title:** StoneAge: Pet World (`스톤에이지 키우기`)  
**Launch:** 2026-03-03 (Android + iOS, global)  
**Scope:** OS × Country high-level performance; KOR deep-dive — CPI/ROAS diagnosis, campaign goal audience comparison, saturation check  
**Author:** Haewon Yum · KOR GDS  
**Date:** 2026-04-09

---

## Sections
- **0** — Setup & Discovery (bundle spot-check, login event validation)
- **1** — OS × Country Performance Snapshot (L14D, recent-weighted)
- **2** — KOR Diagnostic: CPI vs ROAS root cause
- **3** — Campaign Goal & Audience Comparison (KOR)
- **4** — Audience Saturation Check (KOR)
- **5** — Summary & Prioritized Recommendations

## Section 0 — Setup

In [207]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    # Coerce all columns to numeric where possible — handles BQ decimal.Decimal
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass
    print(f'\u2705 {label}: {len(df)} rows')
    return df


In [208]:
# ── Parameters ─────────────────────────────────────────────────────────────
BUNDLE_ANDROID   = 'com.netmarble.stonkey'
BUNDLE_IOS       = '6737408689'            # numeric App Store ID — no 'id' prefix in BQ
BUNDLES          = [BUNDLE_ANDROID, BUNDLE_IOS]
BUNDLE_SQL       = "('com.netmarble.stonkey', '6737408689')"   # for cv / fact_dsp_* tables
MMP_BUNDLE_SQL   = "('com.netmarble.stonkey', 'id6737408689')" # for pb table — iOS uses id-prefix in MMP format

LAUNCH_DATE      = '2026-03-03'
ANALYSIS_DATE   = '2026-04-10'    # fixed reference date — all LXD windows computed relative to this
LOOKBACK_DAYS    = 14
RECENT_DAYS      = 7   # primary focus — post-launch-burst
KOR              = 'KOR'

# Netmarble KPI targets (confirmed)
ROAS_D1_TARGET   = 0.09   # 9% — StoneAge: Pet World

# Set after Section 0c validation
LOGIN_EVENT      = 'login'   # ← HYPOTHESIS — confirm in 0c before use
PURCHASE_EVENT   = 'revenue'            # ← discover in 0c


### 0b — Bundle Spot-Check
Confirm data exists in `fact_dsp_core` for both bundles before proceeding.

In [174]:
q_spot = f"""
SELECT
  product.app_market_bundle   AS bundle,
  campaign.os                 AS os,
  MIN(date_utc)               AS first_date,
  MAX(date_utc)               AS last_date,
  SUM(gross_spend_usd)        AS total_spend,
  SUM(installs)               AS total_installs
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_spot = run_query(q_spot, 'Bundle spot-check')
display(df_spot)

✅ Bundle spot-check: 2 rows


,bundle,os,first_date,last_date,total_spend,total_installs
0,6737408689,IOS,2026-03-03,2026-04-10,140049.182732,10378
1,com.netmarble.stonkey,ANDROID,2026-03-03,2026-04-10,767690.618923,75352


### 0c — Login & Purchase Event Validation

**Step 1:** Check `campaign.kpi_actions` in `fact_dsp_core` — authoritative configured KPI per campaign.  
**Step 2:** Cross-validate event names in cv table with meaningful volume.

In [175]:
# Step 1 — KPI event from fact_dsp_core campaign config
q_kpi = f"""
SELECT DISTINCT
  campaign_id,
  campaign.title      AS campaign_title,
  campaign.os         AS os,
  campaign.country    AS country,
  campaign.goal       AS goal,
  campaign.kpi_actions
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc >= DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
ORDER BY os, country, goal
"""
df_kpi = run_query(q_kpi, 'KPI actions by campaign')
display(df_kpi)

✅ KPI actions by campaign: 202 rows


,campaign_id,campaign_title,os,country,goal,kpi_actions
0,naoP7qVSZwYbv4GY,[NEW]stonekey_launch_moloco_WW1_And_AEO(buy_pe...,ANDROID,ARE,OPTIMIZE_CPA_FOR_APP_UA,buy_pet_lv3
1,KbiHe53hwAPGhNUH,[NEW]stonkey_launch_moloco_WW_1_And_AEO(join_c...,ANDROID,ARE,OPTIMIZE_CPA_FOR_APP_UA,
2,unbUzFob1WCmTrVD,[NEW]stonekey_launch_moloco_WW1_And_tROAS_260303,ANDROID,ARE,OPTIMIZE_ROAS_FOR_APP_UA,
3,unbUzFob1WCmTrVD,[NEW]stonekey_launch_moloco_WW1_And_tROAS_260303,ANDROID,ARG,OPTIMIZE_ROAS_FOR_APP_UA,
4,huSii8ixFBbQjDXt,[NEW]stonekey_launch_moloco_WW3_And_tROAS_260304,ANDROID,ARG,OPTIMIZE_ROAS_FOR_APP_UA,visit_shop
...,...,...,...,...,...,...
197,gzQ7yojuudodWmzn,[CUSC]stonekey_launch_moloco_US_iOS_login_260303,IOS,None,OPTIMIZE_CPA_FOR_APP_UA,login
198,Tl7NLclxzSdjUi6S,[CJPC]stonekey_launch_moloco_JP_iOS_login_260303,IOS,JPN,OPTIMIZE_CPA_FOR_APP_UA,login
199,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,KOR,OPTIMIZE_CPA_FOR_APP_UA,login
200,emorTW8Y48l92YcV,[CTWC]stonekey_launch_moloco_TW_iOS_login_260303,IOS,TWN,OPTIMIZE_CPA_FOR_APP_UA,login


In [176]:
# Step 2 — Distinct event names in cv table (login + purchase candidates)
q_events = f"""
SELECT
  cv.event_pb    AS event_name,
  COUNT(*)            AS event_count
FROM `focal-elf-631.prod_stream_view.cv`
WHERE api.product.app.store_id IN {BUNDLE_SQL}
  AND DATE(timestamp) >= DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL 3 DAY)
GROUP BY 1
ORDER BY 2 DESC
LIMIT 30
"""
df_events = run_query(q_events, 'Event names in cv')
display(df_events)

# ── After reviewing output, update LOGIN_EVENT and PURCHASE_EVENT above ──

✅ Event names in cv: 30 rows


,event_name,event_count
0,visit_shop,158233
1,visit_shop_1st,131809
2,login,121895
3,af_app_opened,109888
4,cdn_download_start,5103
5,cdn_download_finished,4569
6,cdn_download_50p,4566
7,cdn_download_75p,4565
8,cdn_download_25p,4559
9,revenue,4020


---
## Section 1 — OS × Country Performance Snapshot (L14D)

Metrics: CPI · Login CPA · Install-to-Login Rate · D1 ROAS · D7 ROAS  
Login counts sourced from cv table using validated `LOGIN_EVENT`.  
> Post-launch-burst expected — L7D vs prior-7D split shown in 1b to reveal recent trend direction.

### 1a — Aggregate Table (L14D)

In [177]:
# Base performance from fact_dsp_core
q_perf = f"""
SELECT
  campaign.os                          AS os,
  campaign.country                     AS country,
  SUM(gross_spend_usd)                 AS spend,
  SUM(installs)                        AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd),
              SUM(installs))           AS cpi,
  SAFE_DIVIDE(SUM(revenue_d1),
              SUM(gross_spend_usd))    AS roas_d1,
  SAFE_DIVIDE(SUM(revenue_d7),
              SUM(gross_spend_usd))    AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
                   AND DATE('{ANALYSIS_DATE}') - 1
GROUP BY 1, 2
"""
df_perf = run_query(q_perf, 'OS x Country performance (fact_dsp_core)')


✅ OS x Country performance (fact_dsp_core): 99 rows


In [178]:
# Login & install user counts from cv — cohort-based (install cohort → did they log in?)
# Anchors on users who installed in the lookback window, then checks for post-install login.
# This prevents >100% install-to-login from users who installed before the window.
# NOTE: update LOGIN_EVENT / INSTALL_EVENT after 0c validation
INSTALL_EVENT = 'install'  # TODO: confirm in 0c output

q_login = f"""
WITH install_cohort AS (
  -- Users who installed within the lookback window
  SELECT
    req.device.os      AS os,
    req.device.geo.country AS country,
    bid.mtid    AS mtid,
    MIN(DATE(timestamp)) AS install_date
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND cv.event_pb = '{INSTALL_EVENT}'
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
                            AND DATE('{ANALYSIS_DATE}') - 1
  GROUP BY 1, 2, 3
),
login_events AS (
  -- First login per mtid (any time after launch — login may occur after the window ends)
  SELECT
    bid.mtid    AS mtid,
    MIN(DATE(timestamp)) AS first_login_date
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND cv.event_pb = '{LOGIN_EVENT}'
    AND DATE(timestamp) >= '{LAUNCH_DATE}'
  GROUP BY 1
)
SELECT
  i.os,
  i.country,
  COUNT(DISTINCT i.mtid)                                          AS unique_installers,
  COUNT(DISTINCT IF(l.first_login_date >= i.install_date,
                    i.mtid, NULL))                               AS unique_login_users
FROM install_cohort i
LEFT JOIN login_events l USING (mtid)
GROUP BY 1, 2
"""
df_login = run_query(q_login, 'Install-to-login cohort (user-level)')


✅ Install-to-login cohort (user-level): 24 rows


In [179]:
# Merge and compute Login CPA + Install-to-Login Rate (user-level)
df_perf['os_key'] = df_perf['os'].str.upper()
df_login['os_key'] = df_login['os'].str.upper()

df_s1 = df_perf.merge(
    df_login[['os_key', 'country', 'unique_login_users', 'unique_installers']],
    on=['os_key', 'country'], how='left'
)

# Safety: force numeric on all computation columns post-merge
for col in ['spend', 'installs', 'cpi', 'roas_d1', 'roas_d7',
            'unique_login_users', 'unique_installers']:
    df_s1[col] = pd.to_numeric(df_s1[col], errors='coerce')

df_s1['login_cpa']        = df_s1['spend'] / df_s1['unique_login_users']
df_s1['install_to_login'] = df_s1['unique_login_users'] / df_s1['unique_installers']

# KPI flag — D1 ROAS target: 9%
df_s1['kpi_d1'] = df_s1['roas_d1'].apply(
    lambda x: '✅' if pd.notna(x) and float(x) >= ROAS_D1_TARGET else '❌'
)

# Display sorted by spend desc — exclude OS x country pairs with no spend in period
cols = ['os', 'country', 'kpi_d1', 'spend', 'installs', 'cpi', 'login_cpa', 'install_to_login', 'roas_d1', 'roas_d7']
display(
    df_s1.loc[df_s1['spend'] > 0, cols]
    .sort_values('spend', ascending=False)
    .style
    .format({'spend': '${:,.0f}', 'installs': '{:,.0f}', 'cpi': '${:.2f}',
             'login_cpa': '${:.2f}', 'install_to_login': '{:.1%}',
             'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'}, na_rep='—')
    .background_gradient(subset=['cpi'], cmap='RdYlGn_r')
    .background_gradient(subset=['roas_d1'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
    .background_gradient(subset=['roas_d7'], cmap='RdYlGn')
    .apply(lambda row: ['font-weight: bold' if row['country'] == KOR else '' for _ in row], axis=1)
)


,os,country,kpi_d1,spend,installs,cpi,login_cpa,install_to_login,roas_d1,roas_d7
31,ANDROID,KOR,❌,"$68,387","2,757",$24.80,$26.83,92.5%,5.1%,14.3%
96,IOS,KOR,❌,"$14,597",946,$15.43,$16.31,94.6%,3.0%,10.5%
98,ANDROID,IDN,❌,"$13,132","3,445",$3.81,$4.26,89.4%,2.9%,5.1%
58,ANDROID,USA,❌,"$12,321",940,$13.11,$14.24,92.0%,2.7%,4.7%
45,ANDROID,THA,❌,"$6,117",931,$6.57,$7.15,91.8%,4.4%,6.2%
81,ANDROID,PHL,❌,"$5,998","2,384",$2.52,$2.79,90.3%,1.1%,2.2%
52,ANDROID,JPN,❌,"$4,964",327,$15.18,$17.12,88.7%,1.2%,11.1%
46,ANDROID,TWN,❌,"$4,077",191,$21.34,$23.29,91.6%,1.7%,2.8%
63,ANDROID,GBR,❌,"$3,362",783,$4.29,$4.70,91.3%,2.7%,3.1%
23,ANDROID,DEU,❌,"$1,816",877,$2.07,$2.36,87.6%,0.2%,0.4%


### 1a (cont.) — ROAS Decomposition Scatter: CPP × ARPPU by Geo
Geo expansion signal: low CPP + high ARPPU → high ROAS, good candidate.  
`ROAS = ARPPU / CPP` — diagonal reference lines mark D7 ROAS = 10%, 50%, 100%.


In [180]:
# D1 payer counts and revenue by OS x country — cohort-based
# Anchor on install date, pull revenue within D0–D1 of install
# D1 ROAS = D1 ARPPU / CPP_D1  where CPP_D1 = spend / D1_payers
q_payers = f"""
WITH install_cohort AS (
  SELECT
    req.device.os                AS os,
    req.device.geo.country           AS country,
    bid.mtid              AS mtid,
    DATE(cv.install_at_pb)   AS install_date   -- confirmed field
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND cv.event_pb = '{INSTALL_EVENT}'
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
                            AND DATE('{ANALYSIS_DATE}') - 1   -- D1-mature: all installs eligible
),
d1_revenue AS (
  SELECT
    bid.mtid              AS mtid,
    DATE(cv.install_at_pb)   AS install_date,
    CAST(SUM(cv.revenue_usd.amount) AS FLOAT64) AS revenue_d1
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND cv.revenue_usd.amount > 0
    AND DATE(timestamp) BETWEEN '{LAUNCH_DATE}' AND DATE('{ANALYSIS_DATE}') - 1
    AND DATE_DIFF(DATE(timestamp), DATE(cv.install_at_pb), DAY) BETWEEN 0 AND 1
  GROUP BY 1, 2
)
SELECT
  i.os,
  i.country,
  COUNT(DISTINCT IF(r.revenue_d1 > 0, i.mtid, NULL))  AS d1_payers,
  CAST(SUM(r.revenue_d1) AS FLOAT64)                   AS total_d1_revenue
FROM install_cohort i
LEFT JOIN d1_revenue r
  ON  i.mtid         = r.mtid
  AND i.install_date = r.install_date
GROUP BY 1, 2
"""
df_payers = run_query(q_payers, 'D1 payers by geo (cohort-based)')

# Merge with df_s1 and compute D1 CPP / D1 ARPPU
df_payers['os_key'] = df_payers['os'].str.upper()
df_scatter = df_s1.merge(
    df_payers[['os_key', 'country', 'd1_payers', 'total_d1_revenue']],
    on=['os_key', 'country'], how='left'
)
for col in ['d1_payers', 'total_d1_revenue']:
    df_scatter[col] = pd.to_numeric(df_scatter[col], errors='coerce')

# D1 CPP = spend / D1 payers | D1 ARPPU = D1 revenue / D1 payers
df_scatter['cpp']   = df_scatter['spend'] / df_scatter['d1_payers']
df_scatter['arppu'] = df_scatter['total_d1_revenue'] / df_scatter['d1_payers']
df_scatter['roas']  = df_scatter['arppu'] / df_scatter['cpp']   # = D1 revenue / spend = D1 ROAS

# Filter: geos with spend and at least 1 D1 payer
df_scatter = df_scatter[(df_scatter['spend'] > 0) & (df_scatter['d1_payers'] > 0)].copy()
print(f'{len(df_scatter)} geo x OS combinations with D1 payer data')


✅ D1 payers by geo (cohort-based): 24 rows
20 geo x OS combinations with D1 payer data


In [181]:
import numpy as np

# Axis bounds — clip to data range so reference lines don't inflate the scale
x_max = df_scatter['cpp'].max() * 1.15
y_max = df_scatter['arppu'].max() * 1.3
y_min = min(0, df_scatter['arppu'].min() * 1.1)

fig = px.scatter(
    df_scatter,
    x='cpp', y='arppu',
    color='os',
    size='spend',
    size_max=40,
    text='country',
    hover_data={'cpp': ':.2f', 'arppu': ':.2f', 'roas': ':.1%', 'd1_payers': ':,',
                'total_d1_revenue': ':,.1f', 'spend': ':,.0f', 'os': False},
    labels={'cpp': 'D1 CPP — Cost Per D1 Payer (USD)', 'arppu': 'D1 ARPPU (USD)'},
    title='StoneAge: Pet World — D1 ROAS Decomposition by Geo<br>'
          '<sup>Size = spend | Color = OS | D1 ROAS = D1 ARPPU / D1 CPP | KPI = 9%</sup>',
    range_x=[0, x_max],
    range_y=[y_min, y_max],
)
fig.update_traces(textposition='top center', textfont_size=10)

# Reference lines: clip x so they don't extend beyond y_max
for roas_level, label, dash in [(ROAS_D1_TARGET, f'KPI {ROAS_D1_TARGET:.0%}', 'dashdot'),
                                  (0.50, 'ROAS 50%', 'dash'),
                                  (1.00, 'ROAS 100%', 'solid')]:
    x_clip = min(x_max, y_max / roas_level)
    x_line = np.linspace(0, x_clip, 200)
    fig.add_scatter(x=x_line, y=x_line * roas_level,
                    mode='lines', line=dict(dash=dash, color='gray', width=1),
                    name=label, showlegend=True)

fig.update_layout(height=520, legend=dict(orientation='h', y=-0.15))
fig.show()

# Summary table
print('\n── Geo ranking by ROAS (ARPPU / CPP) ──')
display(
    df_scatter[['os', 'country', 'spend', 'd1_payers', 'cpp', 'arppu', 'roas']]
    .sort_values('roas', ascending=False)
    .style
    .format({'spend': '${:,.0f}', 'd1_payers': '{:,.0f}', 'cpp': '${:.2f}', 'arppu': '${:.2f}',
             'roas': '{:.1%}'}, na_rep='—')
    .background_gradient(subset=['roas'], cmap='RdYlGn')
)



── Geo ranking by ROAS (ARPPU / CPP) ──


,os,country,spend,d1_payers,cpp,arppu,roas
79,ANDROID,ARE,$750,3,$250.00,$46.97,18.8%
89,ANDROID,BEL,$664,7,$94.89,$14.31,15.1%
13,ANDROID,PER,$382,9,$42.40,$6.25,14.8%
31,ANDROID,KOR,"$68,387",263,$260.03,$15.29,5.9%
45,ANDROID,THA,"$6,117",64,$95.58,$4.69,4.9%
98,ANDROID,IDN,"$13,132",158,$83.12,$3.18,3.8%
52,ANDROID,JPN,"$4,964",23,$215.82,$7.40,3.4%
58,ANDROID,USA,"$12,321",68,$181.19,$6.12,3.4%
96,IOS,KOR,"$14,597",46,$317.32,$10.13,3.2%
97,ANDROID,PRI,$190,3,$63.39,$1.99,3.1%


### 1b — Daily Trend (L14D): Recent Direction
Split L7D vs prior-7D to distinguish post-burst stabilization from continued decline.

In [182]:
q_daily = f"""
SELECT
  date_utc,
  campaign.os                           AS os,
  SUM(gross_spend_usd)                  AS spend,
  SUM(installs)                         AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd),
              SUM(installs))            AS cpi,
  SAFE_DIVIDE(SUM(revenue_d1),
              SUM(gross_spend_usd))     AS roas_d1,
  SAFE_DIVIDE(SUM(revenue_d7),
              SUM(gross_spend_usd))     AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
                   AND DATE('{ANALYSIS_DATE}') - 1
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_daily = run_query(q_daily, 'Daily trend')


✅ Daily trend: 28 rows


In [183]:
import datetime

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['CPI (USD)', f'D1 ROAS (%) — KPI: {ROAS_D1_TARGET:.0%}', 'D7 ROAS (%)'],
    vertical_spacing=0.08
)
for os_val, grp in df_daily.groupby('os'):
    fig.add_trace(go.Scatter(x=grp['date_utc'], y=grp['cpi'],
                             name=f'{os_val}', mode='lines+markers',
                             legendgroup=os_val), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['date_utc'], y=grp['roas_d1'],
                             name=f'{os_val}', mode='lines+markers',
                             legendgroup=os_val, showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=grp['date_utc'], y=grp['roas_d7'],
                             name=f'{os_val}', mode='lines+markers',
                             legendgroup=os_val, showlegend=False), row=3, col=1)

# KPI target line on D1 ROAS panel
fig.add_hline(y=ROAS_D1_TARGET, line_dash='dash', line_color='red', line_width=1.5,
              annotation_text=f'KPI {ROAS_D1_TARGET:.0%}', annotation_position='right',
              row=2, col=1)

# L7D reference band
cutoff = datetime.date.today() - datetime.timedelta(days=RECENT_DAYS)
for row in [1, 2, 3]:
    fig.add_vrect(x0=cutoff, x1=df_daily['date_utc'].max(),
                  fillcolor='lightblue', opacity=0.15, line_width=0,
                  annotation_text='L7D' if row == 1 else '',
                  annotation_position='top left', row=row, col=1)

fig.update_yaxes(tickformat='.1%', row=2, col=1)
fig.update_yaxes(tickformat='.1%', row=3, col=1)
fig.update_layout(title='StoneAge: Pet World — Daily CPI & ROAS Trend (All Geos)',
                  height=680, legend=dict(orientation='h', y=-0.08))
fig.show()


---
## Section 2 — KOR Diagnostic: CPI or ROAS Problem?

Framework:
```
D1 ROAS  =  D1 ARPU  ×  (1 / CPI)
D1 ARPU  =  D1 I2P  ×  D1 ARPPU
CPI      =  CPM  /  IPM  ×  1000
```
Diagnose which lever is broken before recommending action.  
**KPI: D1 ROAS ≥ 9%** (Netmarble target for StoneAge: Pet World)


### 2a — KOR Summary & Problem Classification (L7D)

In [184]:
q_kor = f"""
SELECT
  campaign.os                           AS os,
  campaign.goal                         AS goal,
  SUM(gross_spend_usd)                  AS spend,
  SUM(installs)                         AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd),
              SUM(installs))            AS cpi,
  SAFE_DIVIDE(SUM(revenue_d1),
              SUM(gross_spend_usd))     AS roas_d1,
  SAFE_DIVIDE(SUM(revenue_d7),
              SUM(gross_spend_usd))     AS roas_d7,
  SAFE_DIVIDE(SUM(revenue_d7),
              SUM(installs))            AS arpu_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                   AND DATE('{ANALYSIS_DATE}') - 1
GROUP BY 1, 2
ORDER BY 1, 3 DESC
"""
df_kor = run_query(q_kor, 'KOR summary L7D')
display(df_kor)

✅ KOR summary L7D: 3 rows


,os,goal,spend,installs,cpi,roas_d1,roas_d7,arpu_d7
0,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,33108.211338,441,75.075309,0.038043,0.065013,4.880873
1,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,15332.819570,1614,9.499888,0.003431,0.012000,0.114002
2,IOS,OPTIMIZE_CPA_FOR_APP_UA,9196.332160,656,14.018799,0.025202,0.033946,0.475888


In [185]:
# KOR classification — D1 ROAS vs confirmed target (9%)
# CPI thresholds: not set by Netmarble — flag outliers relative to OS median instead
df_kor['roas_d1_vs_target'] = df_kor['roas_d1'] - ROAS_D1_TARGET
df_kor['roas_d1_ok']        = df_kor['roas_d1'] >= ROAS_D1_TARGET

def classify(row):
    roas_low = not row['roas_d1_ok']
    # Use median CPI across KOR as CPI benchmark (no hard target from Netmarble)
    cpi_median = df_kor['cpi'].median()
    cpi_high   = row['cpi'] > cpi_median * 1.3   # >30% above median = elevated
    if cpi_high and roas_low:  return '🔴 CPI + ROAS — compound'
    if cpi_high:               return '🟡 CPI issue'
    if roas_low:               return '🟡 ROAS below target (9% D1)'
    return '🟢 Healthy'

df_kor['verdict'] = df_kor.apply(classify, axis=1)

display(
    df_kor[['os', 'goal', 'spend', 'cpi', 'roas_d1', 'roas_d7', 'arpu_d7', 'roas_d1_vs_target', 'verdict']]
    .style
    .format({'spend': '${:,.0f}', 'cpi': '${:.2f}',
             'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}',
             'arpu_d7': '${:.2f}', 'roas_d1_vs_target': '{:+.1%}'}, na_rep='—')
    .applymap(lambda v: 'color: red' if isinstance(v, str) and '🔴' in v else
                        'color: orange' if isinstance(v, str) and '🟡' in v else
                        'color: green' if isinstance(v, str) and '🟢' in v else '',
              subset=['verdict'])
    .background_gradient(subset=['roas_d1'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
)


/var/folders/10/5frw1w6d75v_3v680c6f_p100000gp/T/ipykernel_84763/3825467923.py:24: FutureWarning:

Styler.applymap has been deprecated. Use Styler.map instead.



,os,goal,spend,cpi,roas_d1,roas_d7,arpu_d7,roas_d1_vs_target,verdict
0,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,"$33,108",$75.08,3.8%,6.5%,$4.88,-5.2%,🔴 CPI + ROAS — compound
1,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,"$15,333",$9.50,0.3%,1.2%,$0.11,-8.7%,🟡 ROAS below target (9% D1)
2,IOS,OPTIMIZE_CPA_FOR_APP_UA,"$9,196",$14.02,2.5%,3.4%,$0.48,-6.5%,🟡 ROAS below target (9% D1)


### 2b — CPI Decomposition: CPM vs IPM
Source: `fact_dsp_all` — campaign-level, KOR, L7D  
`CPI = CPM / IPM × 1000`

In [186]:
q_cpi_decomp = f"""
-- fact_dsp_all has impressions (for CPM/IPM) but no campaign.goal
-- Pull goal from fact_dsp_core and join on campaign_id
WITH campaign_meta AS (
  SELECT DISTINCT campaign_id, campaign.goal AS goal
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE product.app_market_bundle IN {BUNDLE_SQL}
    AND campaign.country = '{KOR}'
    AND date_utc >= DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
)
SELECT
  a.campaign_id,
  a.campaign.os                                      AS os,
  m.goal,
  SUM(a.gross_spend_usd)                             AS spend,
  SUM(a.impressions)                                 AS impressions,
  SUM(a.installs)                                    AS installs,
  SAFE_DIVIDE(SUM(a.gross_spend_usd),
              SUM(a.impressions)) * 1000             AS cpm,
  SAFE_DIVIDE(SUM(a.installs),
              SUM(a.impressions)) * 1000             AS ipm,
  SAFE_DIVIDE(SUM(a.gross_spend_usd),
              SUM(a.installs))                       AS cpi
FROM `moloco-ae-view.athena.fact_dsp_all` a
LEFT JOIN campaign_meta m USING (campaign_id)
WHERE a.product.app_market_bundle IN {BUNDLE_SQL}
  AND a.campaign.country = '{KOR}'
  AND DATE(a.timestamp_utc) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                                 AND DATE('{ANALYSIS_DATE}') - 1
GROUP BY 1, 2, 3
HAVING SUM(a.installs) > 0
"""
df_cpi_decomp = run_query(q_cpi_decomp, 'CPI decomposition KOR')


✅ CPI decomposition KOR: 4 rows


In [187]:
fig = px.scatter(
    df_cpi_decomp, x='cpm', y='ipm',
    size='spend', color='os',
    hover_data=['campaign_id', 'goal', 'cpi'],
    title='KOR — CPM vs IPM by Campaign (sized by spend)<br><sup>High CPM + Low IPM = compound CPI problem</sup>',
    labels={'cpm': 'CPM (USD)', 'ipm': 'IPM (installs per 1k imps)'}
)
fig.update_layout(height=480)
fig.show()

### 2c — D1 ROAS Decomposition by Campaign (KOR)

```
D1 ROAS  =  D1 I2P  ×  D1 ARPPU  /  CPI
```

**Part 1** — Campaign-level D1 ROAS from `fact_dsp_core` (covers both iOS + Android)  
**Part 2** — D1 I2P × ARPPU from `cv` (Android only)  
⚠️ **iOS `install` events absent from cv** (confirmed via BQ — 0 rows for iOS bundle). iOS users are not sending install postbacks to the cv pipeline for this title. Part 1 covers both OS; Part 2 is Android-only.


In [188]:
# 2c Part 1 — Campaign-level D1 ROAS breakdown from fact_dsp_core (iOS + Android)
# fact_dsp_core has revenue_d1 directly — use this as primary decomposition
q_roas_camp = f"""
SELECT
  campaign_id,
  campaign.os                                              AS os,
  campaign.goal                                            AS goal,
  SUM(gross_spend_usd)                                     AS spend,
  SUM(installs)                                            AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))         AS cpi,
  SUM(revenue_d1)                                          AS d1_revenue,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))       AS roas_d1,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd))       AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc >= DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
GROUP BY 1, 2, 3
HAVING SUM(gross_spend_usd) > 0
ORDER BY spend DESC
"""
df_roas_camp = run_query(q_roas_camp, 'D1 ROAS by campaign (KOR)')
display(
    df_roas_camp.style
    .format({'spend': '${:,.0f}', 'installs': '{:,.0f}', 'cpi': '${:.2f}',
              'd1_revenue': '${:,.0f}', 'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'},
             na_rep='—')
    .background_gradient(subset=['roas_d1'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
    .apply(lambda row: ['font-weight: bold' if row['os'] == 'IOS' else '' for _ in row], axis=1)
)

# Bar chart: D1 ROAS by campaign with KPI line
labels = df_roas_camp['campaign_id'].str[:14] + '… [' + df_roas_camp['os'] + ']'
fig = go.Figure()
colors = df_roas_camp['os'].map({'ANDROID': '#636EFA', 'IOS': '#EF553B'})
fig.add_bar(x=labels, y=df_roas_camp['roas_d1'],
            marker_color=colors,
            text=df_roas_camp['roas_d1'].apply(lambda v: f'{v:.1%}' if pd.notna(v) else '—'),
            textposition='outside')
fig.add_hline(y=ROAS_D1_TARGET, line_dash='dash', line_color='red',
              annotation_text=f'KPI {ROAS_D1_TARGET:.0%}')
fig.update_layout(height=420, xaxis_tickangle=-30,
    title='KOR — D1 ROAS by Campaign (fact_dsp_core)',
    yaxis_title='D1 ROAS', yaxis_tickformat='.0%')
fig.show()


✅ D1 ROAS by campaign (KOR): 4 rows


,campaign_id,os,goal,spend,installs,cpi,d1_revenue,roas_d1,roas_d7
0,nazpxG3J5MareHRz,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,"$33,409",445,$75.08,"$1,260",3.8%,6.4%
1,yFGQdt2EPPm0NU97,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,"$10,169",734,$13.85,$52,0.5%,1.8%
2,GdPe1hm9tPMaUhbt,IOS,OPTIMIZE_CPA_FOR_APP_UA,"$9,497",663,$14.32,$232,2.4%,3.3%
3,ylgO8XQvDb5nx3k4,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,"$5,164",880,$5.87,$1,0.0%,0.1%


In [189]:
# 2c Part 2 — D1 CPP × D1 ARPPU by campaign (cv, Android, L7D)
# CPP = spend / D1 payers | ARPPU = D1 revenue / D1 payers
# iOS install events absent from cv — Android only (confirmed)
q_camp_payers = f"""
WITH install_cohort AS (
  SELECT
    api.campaign.id        AS campaign_id,
    req.device.os                 AS os,
    bid.mtid               AS mtid,
    DATE(cv.install_at_pb)    AS install_date
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND cv.event_pb = '{INSTALL_EVENT}'
    AND req.device.geo.country = '{KOR}'
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                            AND DATE('{ANALYSIS_DATE}') - 1
),
d1_rev AS (
  SELECT
    api.campaign.id        AS campaign_id,
    bid.mtid               AS mtid,
    DATE(cv.install_at_pb)    AS install_date,
    SUM(cv.revenue_usd.amount)      AS revenue_d1
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE api.product.app.store_id IN {BUNDLE_SQL}
    AND req.device.geo.country = '{KOR}'
    AND cv.revenue_usd.amount > 0
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                            AND DATE('{ANALYSIS_DATE}') - 1
    AND DATE_DIFF(DATE(timestamp), DATE(cv.install_at_pb), DAY) BETWEEN 0 AND 1
  GROUP BY 1, 2, 3
)
SELECT
  i.campaign_id,
  i.os,
  COUNT(DISTINCT i.mtid)                                               AS cv_installs,
  COUNT(DISTINCT IF(r.revenue_d1 > 0, i.mtid, NULL))                  AS d1_payers,
  CAST(SUM(r.revenue_d1) AS FLOAT64)                                  AS total_d1_revenue
FROM install_cohort i
LEFT JOIN d1_rev r
  ON  i.mtid         = r.mtid
  AND i.campaign_id  = r.campaign_id
  AND i.install_date = r.install_date
GROUP BY 1, 2
HAVING COUNT(DISTINCT i.mtid) > 0
"""
df_camp_payers = run_query(q_camp_payers, 'D1 payers by campaign (KOR, cv)')

# Join with Part 1 for spend
df_camp_payers['os_key'] = df_camp_payers['os'].str.upper()
df_roas_camp['os_key'] = df_roas_camp['os'].str.upper()
df_2c = df_roas_camp.merge(
    df_camp_payers[['campaign_id', 'os_key', 'cv_installs', 'd1_payers', 'total_d1_revenue']],
    on=['campaign_id', 'os_key'], how='left'
)
for col in ['cv_installs', 'd1_payers', 'total_d1_revenue']:
    df_2c[col] = pd.to_numeric(df_2c[col], errors='coerce')

df_2c['cpp']   = df_2c['spend'] / df_2c['d1_payers']
df_2c['arppu'] = df_2c['total_d1_revenue'] / df_2c['d1_payers']
df_2c['roas_check'] = df_2c['arppu'] / df_2c['cpp']  # should ≈ roas_d1 from Part 1
df_2c_plot = df_2c[(df_2c['spend'] > 0) & (df_2c['d1_payers'] > 0)].copy()

import numpy as np
x_max = df_2c_plot['cpp'].max() * 1.2
y_max = df_2c_plot['arppu'].max() * 1.3

fig = px.scatter(
    df_2c_plot,
    x='cpp', y='arppu',
    size='spend', size_max=40,
    text=df_2c_plot['campaign_id'].str[:12] + '…',
    hover_data={'cpp': ':.2f', 'arppu': ':.2f', 'roas_check': ':.1%',
                'd1_payers': ':,', 'spend': ':,.0f'},
    labels={'cpp': 'D1 CPP (USD)', 'arppu': 'D1 ARPPU (USD)'},
    title='KOR — D1 ROAS Decomposition by Campaign (cv, Android, L7D)<br>'
          '<sup>Size = spend | D1 ROAS = ARPPU / CPP | KPI = 9%</sup>',
    range_x=[0, x_max], range_y=[0, y_max],
)
fig.update_traces(textposition='top center', textfont_size=9)

for roas_level, label, dash in [(ROAS_D1_TARGET, f'KPI {ROAS_D1_TARGET:.0%}', 'dashdot'),
                                (0.50, 'ROAS 50%', 'dash'),
                                (1.00, 'ROAS 100%', 'solid')]:
    x_clip = min(x_max, y_max / roas_level)
    x_line = np.linspace(0, x_clip, 200)
    fig.add_scatter(x=x_line, y=x_line * roas_level, mode='lines',
                    line=dict(dash=dash, color='gray', width=1),
                    name=label, showlegend=True)

fig.update_layout(height=500, legend=dict(orientation='h', y=-0.15))
fig.show()


✅ D1 payers by campaign (KOR, cv): 4 rows


### 2d — Attributed vs Organic: User Quality Signal
Source: `moloco-dsp-data-view.postback.pb` (unsampled)  
> cv contains attributed events only — pb table needed to compare organic vs Moloco-attributed.  
> ⚠️ Verify pb schema field names before running — paths differ from cv.

**TODO: confirm these pb field names before executing:**
- Attribution flag: `moloco.attributed` (bool) or equivalent
- Bundle: `app.bundle` or `product.app_market_bundle`
- Country: `device.country` or `geo.country`
- Revenue: `revenue_usd.amount` or `revenue.amount`
- Device ID: `device.ifa` or `device.id`

In [223]:
# 2d — Attributed (by campaign) vs Unattributed: D1 + D7 user quality
# Android attributed: split by moloco.campaign_id (directly from pb)
# iOS attributed: non-LAT (device.idfa) + LAT/PA (moloco.mtid.maid.id)
# Unattributed: non-LAT only (device.idfa IS NOT NULL) — LAT unattributed excluded
# User key: COALESCE(device.idfa, moloco.mtid.maid.id) covers both non-LAT and PA-LAT

q_organic = f"""
WITH installs AS (
  SELECT
    device.os                                          AS os,
    CASE
      -- Android attributed: one group per campaign
      WHEN device.os = 'ANDROID' AND attribution.attributed = TRUE
        THEN moloco.campaign_id
      -- iOS attributed non-LAT
      WHEN device.os = 'IOS' AND attribution.attributed = TRUE
           AND device.idfa IS NOT NULL
        THEN 'Moloco (non-LAT)'
      -- iOS attributed LAT with PA synthetic MAID
      WHEN device.os = 'IOS' AND attribution.attributed = TRUE
           AND device.idfa IS NULL AND moloco.mtid.maid.id IS NOT NULL
        THEN 'Moloco (LAT/PA)'
      -- Unattributed non-LAT (both OS)
      WHEN attribution.attributed = FALSE AND device.idfa IS NOT NULL
        THEN 'Unattributed'
    END                                                AS group_label,
    COALESCE(device.idfa, moloco.mtid.maid.id)         AS user_id,
    DATE(event.install_at)                             AS install_date
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE app.bundle IN {MMP_BUNDLE_SQL}
    AND device.country = '{KOR}'
    AND LOWER(event.name) = 'install'
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                            AND DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL 1 DAY)
    AND DATE(event.install_at) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                                   AND DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL 1 DAY)
    -- Exclude unattributed LAT (no user_id available)
    AND (attribution.attributed = TRUE OR device.idfa IS NOT NULL)
),
revenue AS (
  SELECT
    COALESCE(device.idfa, moloco.mtid.maid.id)         AS user_id,
    DATE(event.install_at)                             AS install_date,
    SUM(IF(DATE_DIFF(DATE(timestamp), DATE(event.install_at), DAY) BETWEEN 0 AND 1,
           event.revenue_usd.amount, 0))               AS revenue_d1,
    SUM(event.revenue_usd.amount)                      AS revenue_d7
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE app.bundle IN {MMP_BUNDLE_SQL}
    AND device.country = '{KOR}'
    AND event.revenue_usd.amount > 0
    AND DATE(timestamp) BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                            AND DATE('{ANALYSIS_DATE}') - 1
    AND DATE_DIFF(DATE(timestamp), DATE(event.install_at), DAY) BETWEEN 0 AND 7
    AND COALESCE(device.idfa, moloco.mtid.maid.id) IS NOT NULL
  GROUP BY 1, 2
)
SELECT
  i.os,
  i.group_label,
  COUNT(DISTINCT i.user_id)                                                      AS installs,
  COUNT(DISTINCT IF(r.revenue_d1 > 0, i.user_id, NULL))                         AS d1_payers,
  SAFE_DIVIDE(
    COUNT(DISTINCT IF(r.revenue_d1 > 0, i.user_id, NULL)),
    COUNT(DISTINCT i.user_id))                                                   AS d1_i2p,
  SAFE_DIVIDE(SUM(r.revenue_d1), COUNT(DISTINCT i.user_id))                     AS d1_arpu,
  SAFE_DIVIDE(SUM(r.revenue_d1),
    COUNT(DISTINCT IF(r.revenue_d1 > 0, i.user_id, NULL)))                      AS d1_arppu,
  APPROX_QUANTILES(IF(r.revenue_d1 > 0, r.revenue_d1, NULL), 100)[OFFSET(50)]  AS median_d1_arppu,
  COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL))                         AS d7_payers,
  SAFE_DIVIDE(
    COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL)),
    COUNT(DISTINCT i.user_id))                                                   AS d7_i2p,
  SAFE_DIVIDE(SUM(r.revenue_d7), COUNT(DISTINCT i.user_id))                     AS d7_arpu,
  SAFE_DIVIDE(SUM(r.revenue_d7),
    COUNT(DISTINCT IF(r.revenue_d7 > 0, i.user_id, NULL)))                      AS d7_arppu,
  APPROX_QUANTILES(IF(r.revenue_d7 > 0, r.revenue_d7, NULL), 100)[OFFSET(50)]  AS median_d7_arppu
FROM installs i
LEFT JOIN revenue r
  ON  i.user_id      = r.user_id
  AND i.install_date = r.install_date
WHERE i.group_label IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_organic = run_query(q_organic, 'Attributed by campaign + Unattributed (pb, KOR, L7D)')
display(df_organic.style.format({
    'installs': '{:,.0f}', 'd1_payers': '{:,.0f}', 'd7_payers': '{:,.0f}',
    'd1_i2p': '{:.2%}', 'd7_i2p': '{:.2%}',
    'd1_arpu': '${:.2f}', 'd7_arpu': '${:.2f}',
    'd1_arppu': '${:.2f}', 'd7_arppu': '${:.2f}',
    'median_d1_arppu': '${:.2f}', 'median_d7_arppu': '${:.2f}'
}, na_rep='—'))


✅ Attributed by campaign + Unattributed (pb, KOR, L7D): 7 rows


,os,group_label,installs,d1_payers,d1_i2p,d1_arpu,d1_arppu,median_d1_arppu,d7_payers,d7_i2p,d7_arpu,d7_arppu,median_d7_arppu
0,ANDROID,Unattributed,"30,142",995,3.30%,$3.84,$116.18,$14.70,"1,195",3.96%,$5.78,$145.86,$21.55
1,ANDROID,nazpxG3J5MareHRz,438,88,20.09%,$3.00,$14.95,$4.04,97,22.15%,$4.77,$21.54,$6.10
2,ANDROID,yFGQdt2EPPm0NU97,730,26,3.56%,$0.08,$2.23,$0.96,34,4.66%,$0.25,$5.30,$1.16
3,ANDROID,ylgO8XQvDb5nx3k4,880,3,0.34%,$0.00,$0.32,$0.19,4,0.45%,$0.00,$0.96,$0.19
4,IOS,Moloco (LAT/PA),376,16,4.26%,$0.16,$3.68,$0.96,20,5.32%,$0.32,$6.07,$2.09
5,IOS,Moloco (non-LAT),278,11,3.96%,$0.62,$15.57,$6.68,11,3.96%,$0.65,$16.51,$6.68
6,IOS,Unattributed,306,43,14.05%,$1.74,$12.40,$2.32,51,16.67%,$3.54,$21.25,$4.82


In [225]:
# 2d viz — Per OS, grouped by attribution group (campaign / Moloco / Unattributed)

# ── Campaign goal lookup ─────────────────────────────────────────────────────
q_goals = f"""
SELECT DISTINCT
  CAST(campaign_id AS STRING) AS campaign_id,
  campaign.goal               AS goal
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND date_utc >= '{LAUNCH_DATE}'
"""
df_goals = run_query(q_goals, 'Campaign goals')
goal_map = dict(zip(df_goals['campaign_id'], df_goals['goal']))

def _legend_label(gl):
    """Append goal in parens for Android campaign IDs."""
    if gl in goal_map:
        return f"{gl} ({goal_map[gl]})"
    return gl

# ── Build chart ──────────────────────────────────────────────────────────────
os_list = sorted(df_organic['os'].unique().tolist())
raw_groups = df_organic['group_label'].unique().tolist()

_campaign_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
_cidx = 0
color_map = {}
for gl in sorted(raw_groups):
    if gl == 'Unattributed':
        color_map[gl] = '#EF553B'
    elif gl == 'Moloco (non-LAT)':
        color_map[gl] = '#636EFA'
    elif gl == 'Moloco (LAT/PA)':
        color_map[gl] = '#AB63FA'
    else:
        color_map[gl] = _campaign_palette[_cidx % len(_campaign_palette)]
        _cidx += 1

fig = make_subplots(rows=3, cols=2,
    subplot_titles=['D1 I2P', 'D7 I2P',
                    'D1 ARPPU — Mean (USD)', 'D7 ARPPU — Mean (USD)',
                    'D1 ARPPU — Median (USD)', 'D7 ARPPU — Median (USD)'],
    vertical_spacing=0.15, horizontal_spacing=0.12)

metrics = [
    ('d1_i2p',          1, 1, lambda v: f'{v:.1%}'),
    ('d7_i2p',          1, 2, lambda v: f'{v:.1%}'),
    ('d1_arppu',        2, 1, lambda v: f'${v:.2f}'),
    ('d7_arppu',        2, 2, lambda v: f'${v:.2f}'),
    ('median_d1_arppu', 3, 1, lambda v: f'${v:.2f}'),
    ('median_d7_arppu', 3, 2, lambda v: f'${v:.2f}'),
]

for metric, r, c, fmt_fn in metrics:
    is_first = (r == 1 and c == 1)
    y_all = []
    for label in sorted(raw_groups):
        df_grp = df_organic[df_organic['group_label'] == label].set_index('os')
        y_vals = [df_grp.loc[os, metric] if os in df_grp.index else None for os in os_list]
        y_all += [v for v in y_vals if v is not None and pd.notna(v)]
        fig.add_bar(
            x=os_list, y=y_vals,
            name=_legend_label(label),
            marker_color=color_map.get(label, 'gray'),
            text=[fmt_fn(v) if v is not None and pd.notna(v) else '—' for v in y_vals],
            textposition='outside',
            legendgroup=_legend_label(label),
            showlegend=is_first,
            row=r, col=c
        )
    if y_all:
        fig.update_yaxes(range=[0, max(y_all) * 1.25], row=r, col=c)
    if '%' in fmt_fn(0.1):
        fig.update_yaxes(tickformat='.0%', row=r, col=c)

fig.update_layout(
    height=950, barmode='group',
    legend=dict(orientation='h', y=-0.05),
    title='KOR — Attributed (by campaign) vs Unattributed: D1 & D7 User Quality by OS (pb, L7D)'
)
fig.show()


✅ Campaign goals: 32 rows


---
## Section 3 — Campaign Goal & Audience Analysis (KOR)

### 3a — Campaign Inventory (KOR, all active)

In [ ]:
q_campaigns = f"""
SELECT
  campaign_id,
  campaign.title             AS title,
  campaign.os                AS os,
  campaign.goal              AS goal,
  campaign.kpi_actions,
  SUM(gross_spend_usd)       AS spend_l7d,
  SUM(installs)              AS installs_l7d,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs)) AS cpi_l7d,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd)) AS roas_d7_l7d
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE product.app_market_bundle IN {BUNDLE_SQL}
  AND campaign.country = '{KOR}'
  AND date_utc BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
                   AND DATE('{ANALYSIS_DATE}') - 1
GROUP BY 1, 2, 3, 4, 5
HAVING spend_l7d > 0
ORDER BY spend_l7d DESC
"""
df_campaigns = run_query(q_campaigns, 'KOR campaign inventory')
display(df_campaigns)

### 3b — Performance by Goal Type

In [ ]:
df_by_goal = df_campaigns.groupby(['os', 'goal']).agg(
    spend=('spend_l7d', 'sum'),
    installs=('installs_l7d', 'sum'),
    campaigns=('campaign_id', 'count')
).reset_index()

# Re-derive CPI/ROAS from aggregates (weighted average)
goal_base = df_campaigns.groupby(['os', 'goal']).apply(
    lambda g: pd.Series({
        'cpi': g['spend_l7d'].sum() / g['installs_l7d'].sum() if g['installs_l7d'].sum() > 0 else None,
    })
).reset_index()

df_by_goal = df_by_goal.merge(goal_base, on=['os', 'goal'])
display(df_by_goal)

fig = px.bar(df_by_goal, x='goal', y='spend', color='os', barmode='group',
             title='KOR — Spend by Campaign Goal & OS (L7D)',
             text_auto='.2s')
fig.show()

/var/folders/10/5frw1w6d75v_3v680c6f_p100000gp/T/ipykernel_84763/3359490297.py:8: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,os,goal,spend,installs,campaigns,cpi
0,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,15332.819570,1614,2,9.499888
1,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,33108.211338,441,1,75.075309
2,IOS,OPTIMIZE_CPA_FOR_APP_UA,9196.332160,656,1,14.018799


### 3c — Audience Overlap: Impressed Users by Campaign Goal

Compare device ID pools served impressions by CPI-goal vs ROAS-goal campaigns in KOR.  
⚠️ **Table source for impression-level user data needs confirmation** — verify correct table with BQ Agent (likely `focal-elf-631.prod.trace*` or a bid-win log).  

Steps:
1. Get distinct device IDs served ≥1 impression per goal type
2. Compute Jaccard overlap: `|CPI ∩ ROAS| / |CPI ∪ ROAS|`
3. Compare post-install quality (I2P, ARPPU) for each segment (CPI-only / ROAS-only / overlap)

In [ ]:
# TODO: confirm impression-level table and device ID field
IMP_TABLE      = 'TODO'   # e.g. focal-elf-631.prod.trace*
IMP_DEVICE_COL = 'TODO'   # e.g. device_id or ifa
IMP_CAMP_COL   = 'TODO'   # campaign_id column name
IMP_DATE_COL   = 'TODO'   # date/timestamp column
IMP_COUNTRY_COL= 'TODO'   # country column

# Join campaign goal from df_campaigns to get goal labels per campaign_id
campaign_goals = df_campaigns[['campaign_id', 'goal']].drop_duplicates()

# q_overlap = f"""
# WITH imp_users AS (
#   SELECT DISTINCT
#     {IMP_CAMP_COL}   AS campaign_id,
#     {IMP_DEVICE_COL} AS device_id
#   FROM `{IMP_TABLE}`
#   WHERE {IMP_COUNTRY_COL} = '{KOR}'
#     AND {IMP_DATE_COL} >= DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {RECENT_DAYS} DAY)
#     AND {IMP_CAMP_COL} IN ({','.join(repr(c) for c in campaign_goals['campaign_id'])})
# ),
# goal_users AS (
#   SELECT
#     g.goal,
#     u.device_id
#   FROM imp_users u
#   JOIN <campaign_goal_lookup> g ON u.campaign_id = g.campaign_id
# )
# SELECT
#   goal,
#   COUNT(DISTINCT device_id) AS unique_users
# FROM goal_users
# GROUP BY 1
# """
print('Uncomment after confirming impression table + field names')

In [ ]:
# Overlap calculation (run after populating sets above)
# cpi_users  = set(df_cpi_imp['device_id'])
# roas_users = set(df_roas_imp['device_id'])

# intersection = len(cpi_users & roas_users)
# union        = len(cpi_users | roas_users)
# jaccard      = intersection / union if union > 0 else 0

# print(f'CPI-only users   : {len(cpi_users  - roas_users):,}')
# print(f'ROAS-only users  : {len(roas_users - cpi_users):,}')
# print(f'Overlap (both)   : {intersection:,}')
# print(f'Jaccard similarity: {jaccard:.2%}')
print('Uncomment after pulling impression user sets')

Uncomment after pulling impression user sets


---
## Section 4 — Audience Saturation Check (KOR)

Source: `moloco-ae-view.athena.fact_supply`

```
bid_rate        = bids / bid_requests
win_rate        = bids_won / bids
clear_rate      = impressions / bids_won
bid_to_imp_rate = impressions / bids  =  win_rate × clear_rate
```

Rising CPM + falling bid-to-imp rate → saturation signal. Decompose into win_rate vs clear_rate to isolate cause.

In [193]:
# fact_supply: country is 3-letter uppercase ('KOR'), os is uppercase
q_sat = f"""
SELECT
  date_utc,
  req.os                                                      AS os,
  SUM(bid_requests)                                           AS bid_requests,
  SUM(bids)                                                   AS bids,
  SUM(bids_won)                                               AS bids_won,
  SUM(impressions)                                            AS impressions,
  SUM(media_cost_usd)                                         AS spend,
  SAFE_DIVIDE(SUM(bids), SUM(bid_requests))                  AS bid_rate,
  SAFE_DIVIDE(SUM(bids_won), SUM(bids))                      AS win_rate,
  SAFE_DIVIDE(SUM(impressions), SUM(bids_won))               AS clear_rate,
  SAFE_DIVIDE(SUM(impressions), SUM(bids))                   AS bid_to_imp_rate,
  SAFE_DIVIDE(SUM(media_cost_usd), SUM(impressions)) * 1000  AS cpm
FROM `moloco-ae-view.athena.fact_supply`
WHERE req.country = 'KOR'
  AND date_utc BETWEEN DATE_SUB(DATE('{ANALYSIS_DATE}'), INTERVAL {LOOKBACK_DAYS} DAY)
                   AND DATE('{ANALYSIS_DATE}') - 1
  AND campaign_id IN (SELECT DISTINCT campaign_id
                      FROM `moloco-ae-view.athena.fact_dsp_core`
                      WHERE product.app_market_bundle IN {BUNDLE_SQL}
                        AND date_utc >= '{LAUNCH_DATE}')
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_sat = run_query(q_sat, 'Saturation signals KOR (fact_supply)')

✅ Saturation signals KOR (fact_supply): 28 rows


In [194]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['CPM (USD) — rising = increasing competition',
                    'Bid-to-Imp Rate = win_rate × clear_rate — falling = harder to serve'],
    vertical_spacing=0.12
)
for os_val, grp in df_sat.groupby('os'):
    fig.add_trace(go.Scatter(x=grp['date_utc'], y=grp['cpm'],
                             name=f'{os_val}', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['date_utc'], y=grp['bid_to_imp_rate'],
                             name=f'{os_val}', mode='lines+markers', showlegend=False), row=2, col=1)

fig.update_yaxes(tickformat='.2%', row=2, col=1)
fig.update_layout(title='KOR — Saturation Signals (L14D)', height=550,
                  legend=dict(orientation='h', y=-0.1))
fig.show()

# Decompose bid-to-imp rate
print('\n── Win rate & Clear rate decomposition ──')
display(df_sat[['date_utc', 'os', 'win_rate', 'clear_rate', 'bid_to_imp_rate', 'bid_rate', 'cpm']]
        .tail(14)
        .style.format({'win_rate': '{:.2%}', 'clear_rate': '{:.2%}',
                       'bid_to_imp_rate': '{:.2%}', 'bid_rate': '{:.2%}', 'cpm': '${:.3f}'}))


── Win rate & Clear rate decomposition ──


,date_utc,os,win_rate,clear_rate,bid_to_imp_rate,bid_rate,cpm
14,2026-04-03,ANDROID,36.19%,92.30%,33.40%,nan%,$0.643
15,2026-04-03,IOS,31.65%,71.06%,22.49%,nan%,$0.943
16,2026-04-04,ANDROID,29.36%,90.14%,26.47%,nan%,$1.099
17,2026-04-04,IOS,24.43%,56.04%,13.69%,nan%,$2.550
18,2026-04-05,ANDROID,31.86%,86.87%,27.68%,nan%,$1.319
19,2026-04-05,IOS,23.37%,51.93%,12.13%,nan%,$2.797
20,2026-04-06,ANDROID,33.97%,96.95%,32.93%,nan%,$0.821
21,2026-04-06,IOS,24.40%,59.49%,14.52%,nan%,$1.839
22,2026-04-07,ANDROID,28.71%,98.74%,28.35%,nan%,$1.044
23,2026-04-07,IOS,28.27%,66.21%,18.72%,nan%,$1.266


---
---
## Section 5 — iOS SKAN Performance: Pre vs Post PA Enablement (April 3, 2026 KST)

PA (Probabilistic Attribution) was enabled for StoneAge iOS on **April 3, 2026 KST**.
This section checks whether PA enablement changed measured SKAN performance.

**Source:** `focal-elf-631.standard_report_v1_view.report_final_skan`  
**iOS KOR campaign:** `GdPe1hm9tPMaUhbt`  
**Metrics:** SKAN CPI, SKAN ROAS (min/max revenue bounds), SKAN conversion count  

```
SKAN_CPI  = spend / SKAN_ConversionCount
SKAN_ROAS = SKAN_ConversionEventRevenueMinSum or MaxSum / spend  (SKAN revenue bounds)
```

⚠️ SKAN revenue is reported as a range (min/max) due to conversion value bucketing — use midpoint as estimate.


In [229]:
PA_DATE      = '2026-04-03'  # April 3 KST — PA enabled for StoneAge iOS
PRE_PA_DAYS  = 7             # 7 days before PA as pre-PA window (avoids launch-burst)

# iOS KOR campaign ID confirmed in Section 3a
IOS_KOR_CAMPAIGN = 'GdPe1hm9tPMaUhbt'

SKAN_ANALYSIS_DATE = '2026-04-20'

# Source: report_final_skan — Apple SKAN postback conversion value revenue (min/max bounds).
# NOTE: MCP 'SKAN Revenue' source not yet confirmed — investigate before aligning.
q_skan = f"""
SELECT
  DATE(time_bucket)                                              AS date,
  CASE WHEN DATE(time_bucket) >= '{PA_DATE}'
       THEN 'Post-PA' ELSE 'Pre-PA' END                         AS period,
  SUM(spend)                                                     AS spend,
  SUM(SKAN_ConversionCount)                                      AS skan_conversions,
  SAFE_DIVIDE(SUM(spend), SUM(SKAN_ConversionCount))            AS skan_cpi,
  SUM(SKAN_ConversionEventRevenueMinSum)                         AS skan_rev_min,
  SUM(SKAN_ConversionEventRevenueMaxSum)                         AS skan_rev_max,
  SAFE_DIVIDE(
    (SUM(SKAN_ConversionEventRevenueMinSum) + SUM(SKAN_ConversionEventRevenueMaxSum)) / 2,
    SUM(spend))                                                  AS skan_roas_mid
FROM `focal-elf-631.standard_report_v1_view.report_final_skan`
WHERE campaign_id = '{IOS_KOR_CAMPAIGN}'
  AND DATE(time_bucket) >= DATE_SUB(DATE('{PA_DATE}'), INTERVAL {PRE_PA_DAYS} DAY)
  AND DATE(time_bucket) <  DATE('{SKAN_ANALYSIS_DATE}')
GROUP BY 1, 2
ORDER BY 1
"""
df_skan = run_query(q_skan, f'SKAN pre/post PA — Pre: {PRE_PA_DAYS}d before PA, Post: since {PA_DATE}')

# Period summary
df_skan_summary = (
    df_skan.groupby('period')
    .agg(spend=('spend', 'sum'),
         skan_conversions=('skan_conversions', 'sum'),
         skan_rev_min=('skan_rev_min', 'sum'),
         skan_rev_max=('skan_rev_max', 'sum'))
    .assign(
        skan_cpi=lambda d: d['spend'] / d['skan_conversions'],
        skan_roas_min=lambda d: d['skan_rev_min'] / d['spend'],
        skan_roas_max=lambda d: d['skan_rev_max'] / d['spend'],
        skan_roas_mid=lambda d: (d['skan_rev_min'] + d['skan_rev_max']) / 2 / d['spend'],
    )
    .reset_index()
)
print(f'── Period summary (Pre-PA = {PRE_PA_DAYS}d window, Apple SKAN postback revenue bounds) ──')
display(
    df_skan_summary.style
    .format({'spend': '${:,.0f}', 'skan_conversions': '{:,.0f}',
             'skan_cpi': '${:.2f}', 'skan_rev_min': '${:,.0f}', 'skan_rev_max': '${:,.0f}',
             'skan_roas_min': '{:.1%}', 'skan_roas_max': '{:.1%}', 'skan_roas_mid': '{:.1%}'},
             na_rep='—')
    .background_gradient(subset=['skan_roas_mid'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
)

# Daily chart
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['SKAN CPI (USD)', 'SKAN ROAS (%) — mid = (min+max)/2', 'Spend (USD)'],
    vertical_spacing=0.08, row_heights=[0.35, 0.35, 0.3])

fig.add_scatter(x=df_skan['date'], y=df_skan['skan_cpi'],
                mode='lines+markers', name='SKAN CPI', row=1, col=1)
fig.add_scatter(x=df_skan['date'], y=df_skan['skan_roas_mid'],
                mode='lines+markers', name='SKAN ROAS (mid)', row=2, col=1)
fig.add_scatter(x=df_skan['date'], y=df_skan['skan_rev_min'] / df_skan['spend'],
                mode='lines', name='SKAN ROAS (min)', line=dict(dash='dash', color='orange'),
                row=2, col=1)
fig.add_scatter(x=df_skan['date'], y=df_skan['skan_rev_max'] / df_skan['spend'],
                mode='lines', name='SKAN ROAS (max)', line=dict(dash='dash', color='green'),
                row=2, col=1)

for row in [1, 2, 3]:
    fig.add_vline(x=pd.Timestamp(PA_DATE).timestamp() * 1000, line_dash='dot',
                  line_color='blue', line_width=1.5,
                  annotation_text='PA Enabled', annotation_position='top left',
                  row=row, col=1)

fig.add_hline(y=ROAS_D1_TARGET, line_dash='dash', line_color='red',
              annotation_text=f'KPI {ROAS_D1_TARGET:.0%}', row=2, col=1)

df_skan['bar_color'] = df_skan['date'].astype(str).apply(
    lambda d: '#EF553B' if d >= PA_DATE else '#636EFA')
fig.add_bar(x=df_skan['date'], y=df_skan['spend'],
            name='Spend', marker_color=df_skan['bar_color'],
            showlegend=False, row=3, col=1)
fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=3, col=1)
fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_yaxes(range=[0, 0.20], row=2, col=1)
fig.update_layout(height=650,
    title=f'StoneAge iOS KOR — SKAN CPI & ROAS: Pre-PA (L{PRE_PA_DAYS}D) vs Post-PA')
fig.show()


✅ SKAN pre/post PA — Pre: 7d before PA, Post: since 2026-04-03: 24 rows
── Period summary (Pre-PA = 7d window, Apple SKAN postback revenue bounds) ──


,period,spend,skan_conversions,skan_rev_min,skan_rev_max,skan_cpi,skan_roas_min,skan_roas_max,skan_roas_mid
0,Post-PA,"$13,608",925,$146,$398,$14.71,1.1%,2.9%,2.0%
1,Pre-PA,"$5,400",341,$238,$802,$15.84,4.4%,14.9%,9.6%


### 5b — MMP Performance: Pre vs Post PA (fact_dsp_core)

Compare CPI and D1 ROAS from MMP-attributed perspective (Moloco's own measurement) over the same pre/post PA windows used in 5a.

- **Pre-PA:** `[PA_DATE − PRE_PA_DAYS, PA_DATE − 1]`  
- **Post-PA:** `[PA_DATE, ANALYSIS_DATE − 1]`  
- Source: `fact_dsp_core` — reflects MMP-attributed installs and revenue (vs SKAN's probabilistic view)


In [230]:
# ── Period summary ──────────────────────────────────────────────────────────
q_mmp_pa = f"""
SELECT
  CASE WHEN date_utc >= DATE('{PA_DATE}') THEN 'Post-PA' ELSE 'Pre-PA' END AS period,
  SUM(gross_spend_usd)                                          AS spend,
  SUM(installs)                                                 AS installs,
  SUM(impressions)                                              AS impressions,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))              AS cpi,
  SAFE_DIVIDE(SUM(impressions), SUM(installs)) * 1000           AS ipm,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(impressions)) * 1000    AS cpm,
  SUM(revenue_d1)                                               AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))            AS roas_d1,
  SUM(revenue_d7)                                               AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd))            AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{IOS_KOR_CAMPAIGN}'
  AND date_utc >= DATE_SUB(DATE('{PA_DATE}'), INTERVAL {PRE_PA_DAYS} DAY)
  AND date_utc <  DATE('{SKAN_ANALYSIS_DATE}')
GROUP BY 1
ORDER BY 1 DESC
"""
df_mmp_pa = run_query(q_mmp_pa, 'MMP period summary pre/post PA (iOS KOR)')

display(
    df_mmp_pa.style
    .format({'spend': '${:,.0f}', 'installs': '{:,.0f}', 'impressions': '{:,.0f}',
             'revenue_d1': '${:,.0f}', 'revenue_d7': '${:,.0f}',
             'cpi': '${:.2f}', 'cpm': '${:.2f}', 'ipm': '{:.1f}',
             'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'}, na_rep='—')
    .background_gradient(subset=['roas_d1'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
)

# ── Daily query ──────────────────────────────────────────────────────────────
q_mmp_daily = f"""
SELECT
  date_utc                                                       AS date,
  CASE WHEN date_utc >= DATE('{PA_DATE}') THEN 'Post-PA' ELSE 'Pre-PA' END AS period,
  SUM(gross_spend_usd)                                          AS spend,
  SUM(installs)                                                 AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))              AS cpi,
  SUM(revenue_d1)                                               AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))            AS roas_d1
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{IOS_KOR_CAMPAIGN}'
  AND date_utc >= DATE_SUB(DATE('{PA_DATE}'), INTERVAL {PRE_PA_DAYS} DAY)
  AND date_utc <  DATE('{SKAN_ANALYSIS_DATE}')
GROUP BY 1, 2
ORDER BY 1
"""
df_mmp_daily = run_query(q_mmp_daily, 'MMP daily CPI & D1 ROAS pre/post PA (iOS KOR)')

# ── Daily line chart ─────────────────────────────────────────────────────────
from plotly.subplots import make_subplots

fig_mmp = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['CPI (USD)', 'D1 ROAS (%)', 'Spend (USD)'],
    vertical_spacing=0.08, row_heights=[0.35, 0.35, 0.3])

df_mmp_daily['date'] = df_mmp_daily['date'].astype(str)  # align with PA_DATE string type
fig_mmp.add_scatter(x=df_mmp_daily['date'], y=df_mmp_daily['cpi'],
                    mode='lines+markers', name='CPI', row=1, col=1)
fig_mmp.add_scatter(x=df_mmp_daily['date'], y=df_mmp_daily['roas_d1'],
                    mode='lines+markers', name='D1 ROAS', row=2, col=1)

for row in [1, 2, 3]:
    fig_mmp.add_shape(type='line', xref='x', yref='paper',
                      x0=PA_DATE, x1=PA_DATE, y0=0, y1=1,
                      line=dict(dash='dot', color='blue', width=1.5),
                      row=row, col=1)
    fig_mmp.add_annotation(x=PA_DATE, y=1.02, xref='x', yref='paper',
                           text='PA Enabled', showarrow=False,
                           font=dict(color='blue', size=10), xanchor='right',
                           row=row, col=1)

df_mmp_daily['bar_color'] = df_mmp_daily['date'].apply(
    lambda d: '#EF553B' if d >= PA_DATE else '#636EFA')
fig_mmp.add_bar(x=df_mmp_daily['date'], y=df_mmp_daily['spend'],
                name='Spend', marker_color=df_mmp_daily['bar_color'],
                showlegend=False, row=3, col=1)
fig_mmp.update_yaxes(tickprefix='$', tickformat=',.0f', row=3, col=1)
fig_mmp.add_hline(y=ROAS_D1_TARGET, line_dash='dash', line_color='red',
                  annotation_text=f'KPI {ROAS_D1_TARGET:.0%}', row=2, col=1)
fig_mmp.update_yaxes(tickformat='.0%', row=2, col=1)
fig_mmp.update_yaxes(range=[0, 0.20], row=2, col=1)
fig_mmp.update_layout(height=650,
    title=f'iOS KOR — MMP Daily CPI & D1 ROAS: Pre-PA (L{PRE_PA_DAYS}D) vs Post-PA')
fig_mmp.show()


✅ MMP period summary pre/post PA (iOS KOR): 2 rows


,period,spend,installs,impressions,cpi,ipm,cpm,revenue_d1,roas_d1,revenue_d7,roas_d7
0,Pre-PA,"$5,400",290,"4,819,667",$18.62,16619541.4,$1.12,$209,3.9%,"$1,223",22.6%
1,Post-PA,"$13,608",964,"8,378,177",$14.12,8691055.0,$1.62,$347,2.5%,$714,5.2%


✅ MMP daily CPI & D1 ROAS pre/post PA (iOS KOR): 24 rows


### 5c — MMP Performance by Traffic Type: LAT vs Non-LAT

Split the iOS KOR campaign by `campaign.is_lat` to see whether LAT (Limit Ad Tracking) or non-LAT traffic drives the ROAS difference pre/post PA.

PA enablement primarily affects **non-LAT** traffic — those users have device IDs visible to the attribution pipeline. LAT traffic relies on SKAN only and should be unaffected by PA. Comparing the two segments isolates the PA signal from background noise.


In [231]:
# ── Period × LAT summary ────────────────────────────────────────────────────
q_lat = f"""
SELECT
  CASE WHEN date_utc >= DATE('{PA_DATE}') THEN 'Post-PA' ELSE 'Pre-PA' END AS period,
  CASE WHEN campaign.is_lat THEN 'LAT' ELSE 'Non-LAT' END                 AS traffic_type,
  SUM(gross_spend_usd)                                                     AS spend,
  SUM(installs)                                                            AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))                         AS cpi,
  SUM(revenue_d1)                                                          AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))                       AS roas_d1,
  SUM(revenue_d7)                                                          AS revenue_d7,
  SAFE_DIVIDE(SUM(revenue_d7), SUM(gross_spend_usd))                       AS roas_d7
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{IOS_KOR_CAMPAIGN}'
  AND date_utc >= DATE_SUB(DATE('{PA_DATE}'), INTERVAL {PRE_PA_DAYS} DAY)
  AND date_utc <  DATE('{SKAN_ANALYSIS_DATE}')
GROUP BY 1, 2
ORDER BY 1 DESC, 2
"""
df_lat = run_query(q_lat, 'MMP LAT vs Non-LAT summary (iOS KOR)')

display(
    df_lat.style
    .format({'spend': '${:,.0f}', 'installs': '{:,.0f}',
             'revenue_d1': '${:,.0f}', 'revenue_d7': '${:,.0f}',
             'cpi': '${:.2f}', 'roas_d1': '{:.1%}', 'roas_d7': '{:.1%}'}, na_rep='—')
    .background_gradient(subset=['roas_d1'], cmap='RdYlGn', vmin=0, vmax=ROAS_D1_TARGET * 2)
)

# ── Daily query by LAT type ──────────────────────────────────────────────────
q_lat_daily = f"""
SELECT
  date_utc                                                                 AS date,
  CASE WHEN campaign.is_lat THEN 'LAT' ELSE 'Non-LAT' END                 AS traffic_type,
  SUM(gross_spend_usd)                                                     AS spend,
  SUM(installs)                                                            AS installs,
  SAFE_DIVIDE(SUM(gross_spend_usd), SUM(installs))                         AS cpi,
  SUM(revenue_d1)                                                          AS revenue_d1,
  SAFE_DIVIDE(SUM(revenue_d1), SUM(gross_spend_usd))                       AS roas_d1
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE campaign_id = '{IOS_KOR_CAMPAIGN}'
  AND date_utc >= DATE_SUB(DATE('{PA_DATE}'), INTERVAL {PRE_PA_DAYS} DAY)
  AND date_utc <  DATE('{SKAN_ANALYSIS_DATE}')
GROUP BY 1, 2
ORDER BY 1, 2
"""
df_lat_daily = run_query(q_lat_daily, 'MMP daily LAT vs Non-LAT (iOS KOR)')
df_lat_daily['date'] = df_lat_daily['date'].astype(str)

# ── Daily line chart: LAT vs Non-LAT ─────────────────────────────────────────
from plotly.subplots import make_subplots

traffic_types = df_lat_daily['traffic_type'].unique().tolist()
colors = {'Non-LAT': '#636EFA', 'LAT': '#EF553B'}

fig_lat = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['CPI (USD)', 'D1 ROAS (%)', 'Spend (USD)'],
    vertical_spacing=0.08, row_heights=[0.35, 0.35, 0.3])

for tt in traffic_types:
    df_tt = df_lat_daily[df_lat_daily['traffic_type'] == tt]
    color = colors.get(tt, 'gray')
    fig_lat.add_scatter(x=df_tt['date'], y=df_tt['cpi'],
                        mode='lines+markers', name=tt, marker_color=color,
                        legendgroup=tt, row=1, col=1)
    fig_lat.add_scatter(x=df_tt['date'], y=df_tt['roas_d1'],
                        mode='lines+markers', name=tt, marker_color=color,
                        legendgroup=tt, showlegend=False, row=2, col=1)
    fig_lat.add_bar(x=df_tt['date'], y=df_tt['spend'],
                    name=tt, marker_color=color, opacity=0.7,
                    legendgroup=tt, showlegend=False, row=3, col=1)

for row in [1, 2, 3]:
    fig_lat.add_shape(type='line', xref='x', yref='paper',
                      x0=PA_DATE, x1=PA_DATE, y0=0, y1=1,
                      line=dict(dash='dot', color='blue', width=1.5),
                      row=row, col=1)
    fig_lat.add_annotation(x=PA_DATE, y=1.02, xref='x', yref='paper',
                            text='PA Enabled', showarrow=False,
                            font=dict(color='blue', size=10), xanchor='right',
                            row=row, col=1)

fig_lat.add_hline(y=ROAS_D1_TARGET, line_dash='dash', line_color='red',
                  annotation_text=f'KPI {ROAS_D1_TARGET:.0%}', row=2, col=1)
fig_lat.update_yaxes(tickformat='.0%', row=2, col=1)
fig_lat.update_yaxes(range=[0, 0.20], row=2, col=1)
fig_lat.update_yaxes(tickprefix='$', tickformat=',.0f', row=3, col=1)
fig_lat.update_layout(height=650, barmode='stack',
    legend=dict(orientation='h', y=-0.05),
    title=f'iOS KOR — MMP CPI & D1 ROAS by LAT Type: Pre-PA (L{PRE_PA_DAYS}D) vs Post-PA')
fig_lat.show()


✅ MMP LAT vs Non-LAT summary (iOS KOR): 3 rows


,period,traffic_type,spend,installs,cpi,revenue_d1,roas_d1,revenue_d7,roas_d7
0,Pre-PA,Non-LAT,"$5,400",290,$18.62,$209,3.9%,"$1,223",22.6%
1,Post-PA,LAT,"$6,778",505,$13.42,$255,3.8%,$529,7.8%
2,Post-PA,Non-LAT,"$6,830",459,$14.88,$92,1.3%,$185,2.7%


✅ MMP daily LAT vs Non-LAT (iOS KOR): 41 rows


---
## Section 6 — Summary & Prioritized Recommendations

**Analysis date:** 2026-04-10 | **KOR focus window:** L7D (Apr 3–Apr 9)  
**iOS note:** CV pipeline has no install events for this title — iOS cohort metrics (ARPPU, I2P) are unavailable; iOS analysis relies on `fact_dsp_core` ROAS and SKAN data.

---

### Status Dashboard (KOR)

| Signal | Android | iOS | Status |
|--------|---------|-----|--------|
| CPI efficiency | $75 (ROAS goal) / $9.50 (CPA goal) | $14 | 🔴 ROAS-goal CPI extremely elevated |
| D1 ROAS vs 9% target | 3.8% (ROAS) / 0.3% (CPA) | 2.5% | 🔴 All campaigns miss target |
| User quality vs organic | ARPPU $11.75 vs organic $116 (10× gap) | Insufficient cv data | 🔴 Android severe quality gap |
| Campaign goal alignment | 2× CPA campaigns near-zero ROAS | 1× CPA campaign | 🔴 CPA campaigns destroying ROAS |
| Audience saturation | Win rate 29–46%, CPM stable | CPM volatile ($0.65–$2.80) | 🟡 iOS CPM — monitor daily |
| SKAN pre vs post PA | — | Pre: 9.6% ✅ → Post: 2.1% (maturing) | 🟡 Post-PA data incomplete — revisit Apr 17 |

---

### Root Cause: Android KOR

```
D1 ROAS  =  D1 ARPU  ×  (1 / CPI)
ROAS-goal:  3.8%  =  $2.85 ARPU  ×  (1 / $75)
CPA-goal:   0.3%  =  $0.28 ARPU  ×  (1 / $9.50)
```

**ROAS-goal campaign (nazpxG3J5MareHRz):** CPI is the bottleneck. At $75 CPI, 9% D1 ROAS requires D1 ARPU ≥ $6.75 — more than 2× current ($2.85). Likely CPM pressure in KOR gaming supply; creative/bid adjustments needed before scale-up.

**CPA-goal campaigns (yFGQdt2EPPm0NU97, ylgO8XQvDb5nx3k4):** CPI is cheap ($5.87–$13.85) but D1 ROAS is 0.0–0.5%. These campaigns maximize install volume but attract non-monetizing users — the exact inverse of what a ROAS-sensitive title needs.

**User quality signal:** Moloco Android I2P (5.7%) is higher than organic (3.3%) — Moloco finds proportionally more payers. But Moloco ARPPU ($11.75) is 10× lower than organic ($116.18). Moloco is reaching the low-ticket tail of the monetization distribution. The high-value audience exists (organic ARPPU confirms it) but Moloco's current signals are not targeting them.

---

### Root Cause: iOS KOR

- D1 ROAS 2.5% vs 9% target. Only 1 CPA-goal campaign (GdPe1hm9tPMaUhbt), 663 installs L7D.
- SKAN pre-PA ROAS was 9.6% (above target) — strongest positive signal in this analysis.
- Post-PA SKAN data is still maturing; SKAN attribution lag (24–72h) inflates CPI and deflates ROAS in recent days.
- No conclusion possible until post-PA cohort matures (~Apr 17).

---

### Prioritized Recommendations

**1. 🔴 Pause ylgO8XQvDb5nx3k4 (Android CPA, $5.87 CPI, 0.0% D1 ROAS) — immediate**
Every dollar here returns ~$0. No path to 9% D1 ROAS at this user quality. Redirect $5K/week to ROAS-goal campaign.

**2. 🔴 Sharply reduce yFGQdt2EPPm0NU97 (Android CPA, $9.50 CPI, 0.3% D1 ROAS)**
30× below KPI target. Keep at minimal floor only if used for audience signal collection. Do not scale.

**3. 🔴 Address ROAS-goal CPI ($75) before increasing budget**
Check Section 2b: if CPM-driven → publisher exclusions or bid cap; if IPM-driven → creative refresh.  
At $75 CPI, even achieving 9% D1 ROAS requires ARPU improvement that the current creative/audience mix cannot deliver.  
Consider adding `visit_shop` or `buy_pet_lv3` as a KPI event to pull higher-intent audience signals into the model.

**4. 🟡 Re-evaluate iOS PA impact on Apr 17 (post-PA SKAN data matures)**
Pre-PA SKAN ROAS of 9.6% is the single strongest performance signal in this notebook. If post-PA data confirms the same or better level after maturation, iOS is the geo to scale — not KOR Android.

**5. 🟡 Bring organic ARPPU gap ($116 vs $11.75) to Netmarble discussion**
This is not necessarily a Moloco failure — organic RPG players who self-discover a game skew toward high-intent whales. The question is whether Moloco can narrow this gap via audience signal improvement, or whether the gap reflects an inherent ceiling on paid acquisition for this genre.


In [ ]:
summary = {
    'CPI efficiency (KOR)':          '⬜ TBD — fill after Section 2b',
    'D7 ROAS (KOR)':                 '⬜ TBD — fill after Section 2a',
    'I2P / ARPPU (attributed)':      '⬜ TBD — fill after Section 2c',
    'Moloco vs organic user quality': '⬜ TBD — fill after Section 2d',
    'Campaign goal alignment':       '⬜ TBD — fill after Section 3b',
    'Audience overlap (CPI/ROAS)':   '⬜ TBD — fill after Section 3c',
    'Saturation risk (KOR)':         '⬜ TBD — fill after Section 4',
}

print('=' * 60)
print('StoneAge: Pet World — KOR Performance Summary')
print('=' * 60)
for k, v in summary.items():
    print(f'  {v:<6} {k}')

print('\n── Top 3 Actions (fill after analysis) ──')
actions = [
    '1. TODO',
    '2. TODO',
    '3. TODO',
]
for a in actions:
    print(f'  {a}')